# Open VCV — Training Local (RTX 3060)

## Tốc độ ước tính (RTX 3060 12GB, AMP FP16)

| Config | Params | VRAM | Thời gian/epoch ImageNet |
|---|---|---|---|
| Full model, img=224 | 7.2M | 7.8GB | ~7.4h |
| Full model, img=128 | 7.2M | 5.1GB | ~2.4h |
| **Small model, img=128** | **512K** | **3.4GB** | **~24 phút** |
| Small + torch.compile | 512K | 3.4GB | ~18 phút |

**Khuyến nghị local:** dùng small model + img=128 để iterate nhanh.  
Khi muốn train full quality: đẩy lên Kaggle T4x2 hoặc TPU.


In [ ]:
# ── Cell 1: Config ──────────────────────────────────────────────────────────
import sys, os, torch
sys.path.insert(0, '/home/bigkatoan/open_vcv')
os.chdir('/home/bigkatoan/open_vcv')

from src.models.VAE import VAE
from src.losses.losses import IntersectUnionLoss
from src.trainers.trainer import Trainer, TrainConfig

print(f'torch {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory//1024**3}GB)')
print('Imports OK')


In [ ]:
# ── Cell 2: Train — Small model, ImageNet 128px (~ 24 phút/epoch) ────────────
# Để train Full model: dùng notebook kaggle_t4x2.ipynb trên Kaggle T4x2

IMAGENET_TRAIN = 'data/imagenet'  # thay đường dẫn nếu ImageNet ở chỗ khác

cfg = TrainConfig(
    run_name      = 'gated_vae_imagenet_local_small',
    dataset       = 'imagenet',
    data_root     = IMAGENET_TRAIN,
    img_size      = 128,          # 224→128: 3x ít VRAM, 3x nhanh hơn
    q             = 2,
    k             = 0,
    batch_size    = 256,
    grad_accum    = 1,
    num_workers   = 4,
    prefetch_factor = 4,
    device        = 'cuda',
    use_amp       = True,
    compile_model = True,         # torch.compile: +20-40% speedup
    use_vae       = False,        # skip decoder, chỉ train contrastive
    total_epochs  = 100,
    lr            = 3e-4,
    log_iter_every = 50,
)

# Small model: 512K params, 3.4GB VRAM, ~24 phút/epoch trên RTX 3060
model = VAE(
    s1_out=8,  s1_heads=4, s1_blocks=1,
    s2_out=8,  s2_heads=4, s2_blocks=1,
    s3_out=8,  s3_heads=8, s3_blocks=1,
    latent_ch=512,
    dec_ch3=64, dec_ch2=32, dec_ch1=16,
    dim_inter=256, dim_unique=256,
    feat_dim=32, hidden_dim=128,
)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model params: {n:,}  (~{n/1e6:.1f}M)')

torch.cuda.empty_cache()
loss_fn = IntersectUnionLoss()
Trainer(model, loss_fn, cfg).train()


In [ ]:
# ── Cell 3 (optional): Full model với COCO (~2 phút/epoch) ───────────────────
# COCO có 118K ảnh → nhanh hơn ImageNet 10x
# Dùng để kiểm tra convergence trước khi train full ImageNet

cfg_coco = TrainConfig(
    run_name      = 'gated_vae_coco_local',
    dataset       = 'coco',
    data_root     = 'data/coco2017',
    img_size      = 128,
    q             = 2,
    k             = 0,
    batch_size    = 64,
    grad_accum    = 1,
    num_workers   = 4,
    device        = 'cuda',
    use_amp       = True,
    compile_model = True,
    use_vae       = False,
    total_epochs  = 50,
    lr            = 3e-4,
    log_iter_every = 20,
)

model_coco = VAE(
    s1_out=16, s1_heads=4,  s1_blocks=1,
    s2_out=16, s2_heads=8,  s2_blocks=2,
    s3_out=16, s3_heads=16, s3_blocks=2,
    latent_ch=2048, dec_ch3=128, dec_ch2=64, dec_ch1=32,
    dim_inter=1024, dim_unique=1024, feat_dim=64, hidden_dim=256,
)
n = sum(p.numel() for p in model_coco.parameters())
print(f'Full model params: {n:,}  (~{n/1e6:.1f}M)')

torch.cuda.empty_cache()
loss_fn2 = IntersectUnionLoss()
Trainer(model_coco, loss_fn2, cfg_coco).train()


In [ ]:
# ── Cell 4: Plot training curves ─────────────────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt, glob, os

EXP_DIRS = sorted(glob.glob('experiments/gated_vae_*local*'))
if not EXP_DIRS:
    print('No experiment found')
else:
    exp = EXP_DIRS[-1]
    print(f'Experiment: {exp}')
    df = pd.read_csv(f'{exp}/losses.csv')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for k in ['union', 'sparse', 'ortho', 'neg']:
        if k in df.columns:
            axes[0].plot(df['epoch'], df[k], label=k)
    axes[0].set_title('Losses'); axes[0].legend()
    if os.path.exists(f'{exp}/metrics.csv'):
        mdf = pd.read_csv(f'{exp}/metrics.csv')
        for k in ['union_consistency', 'sparse_divergence', 'ortho_score']:
            if k in mdf.columns:
                axes[1].plot(mdf['epoch'], mdf[k], label=k)
        axes[1].set_title('Metrics'); axes[1].legend()
    plt.tight_layout(); plt.show()
